# Pertemuan 3 — Data Cleaning: Missing Values, Outlier & Ekstraksi Data

**Nama Lengkap** : Muhammad Zacky Kurniawan  
**NIM** : 240401010217  
**Kelas** : IF403

<a href="https://colab.research.google.com/github/zackykurniawan/data-science-2026/blob/main/Pertemuan3_MuhammadZackyKurniawan_240401010217.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Import Library & Load Dataset

In [20]:
# Import semua library yang diperlukan
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests
from pandas import json_normalize

print('Library berhasil diimport!')

Library berhasil diimport!


In [21]:
# Muat dataset
# Jika menggunakan Google Colab, upload file housing_dirty.csv terlebih dahulu
# atau gunakan URL Google Drive berikut:
# url = 'https://drive.google.com/uc?id=1LfQWProB0VjWN5q8bKuRIgnstULfIRo'
# df = pd.read_csv(url)

df = pd.read_csv('housing_dirty.csv')

print('Dataset berhasil dimuat!')

Dataset berhasil dimuat!


---
## Eksplorasi Awal



In [22]:
# Tampilkan 5 baris pertama
print('=== HEAD (5 baris pertama) ===')
df.head()

=== HEAD (5 baris pertama) ===


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


In [23]:
# Informasi struktur dataset
print('=== INFO DATASET ===')
print(f'Shape awal: {df.shape}')
print()
df.info()

=== INFO DATASET ===
Shape awal: (130, 7)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


In [24]:
# Statistik deskriptif
print('=== STATISTIK DESKRIPTIF ===')
df.describe()

=== STATISTIK DESKRIPTIF ===


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [25]:
# Cek missing values per kolom
print('=== MISSING VALUES PER KOLOM ===')
missing_count = df.isnull().sum()
missing_pct   = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Jumlah Missing' : missing_count,
    'Persentase (%)'  : missing_pct
})
print(missing_summary)

print(f'\nTotal nilai missing: {df.isnull().sum().sum()}')

=== MISSING VALUES PER KOLOM ===
              Jumlah Missing  Persentase (%)
id                         0            0.00
luas_m2                   18           13.85
harga_juta                17           13.08
kota                       0            0.00
kamar                     10            7.69
tahun_bangun               0            0.00
kondisi                    0            0.00

Total nilai missing: 45


In [26]:
# Cek data duplikat
print('=== CEK DUPLIKAT ===')
n_dup = df.duplicated().sum()
print(f'Jumlah baris duplikat: {n_dup} dari {len(df)} total baris')

=== CEK DUPLIKAT ===
Jumlah baris duplikat: 0 dari 130 total baris


---
## Hapus Baris Duplikat

In [27]:
# Tampilkan baris duplikat sebelum dihapus
df_dup = df[df.duplicated(keep=False)]
print(f'Baris duplikat yang ditemukan:')
print(df_dup)

# Hapus duplikat — pertahankan kemunculan pertama
df.drop_duplicates(inplace=True)

print(f'\nShape setelah hapus duplikat: {df.shape}')
print(f'Sisa duplikat: {df.duplicated().sum()}')

Baris duplikat yang ditemukan:
Empty DataFrame
Columns: [id, luas_m2, harga_juta, kota, kamar, tahun_bangun, kondisi]
Index: []

Shape setelah hapus duplikat: (130, 7)
Sisa duplikat: 0


---
## Normalisasi String

In [28]:
# Cek nilai unik sebelum normalisasi
print('Nilai unik kolom [kota] SEBELUM normalisasi:')
print(df['kota'].unique())

print('\nNilai unik kolom [kondisi] SEBELUM normalisasi:')
print(df['kondisi'].unique())

Nilai unik kolom [kota] SEBELUM normalisasi:
['jogja' 'Medan' 'Depok' 'YGY' 'Jakarta' 'jakarta' 'Yogyakarta' 'Bandung'
 'Surabaya' 'dpk' 'sby' 'Makassar' 'mdn' 'medan' 'Semarang' 'semarang'
 'yogyakarta' 'Jogja' 'JAKARTA' 'Smg' 'DEPOK' 'Bdg' 'makassar' 'surabaya'
 'MAKASSAR' 'depok' 'bandung' 'Bandung ' 'SURABAYA' 'Mksr' ' Jakarta']

Nilai unik kolom [kondisi] SEBELUM normalisasi:
['baik' 'Bagus' 'Sedang' 'baik sekali' 'SEDANG' 'sedang' 'BAIK' 'rusak'
 'cukup' 'Baik' 'Cukup' 'perlu renovasi' 'bagus' 'jelek' 'RUSAK']


In [29]:
# Normalisasi: strip whitespace + title case untuk kolom kota
df['kota']    = df['kota'].str.strip().str.title()

# Normalisasi: strip whitespace + lowercase untuk kolom kondisi
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print('Nilai unik kolom [kota] SETELAH normalisasi:')
print(df['kota'].unique())

print('\nNilai unik kolom [kondisi] SETELAH normalisasi:')
print(df['kondisi'].unique())

Nilai unik kolom [kota] SETELAH normalisasi:
['Jogja' 'Medan' 'Depok' 'Ygy' 'Jakarta' 'Yogyakarta' 'Bandung' 'Surabaya'
 'Dpk' 'Sby' 'Makassar' 'Mdn' 'Semarang' 'Smg' 'Bdg' 'Mksr']

Nilai unik kolom [kondisi] SETELAH normalisasi:
['baik' 'bagus' 'sedang' 'baik sekali' 'rusak' 'cukup' 'perlu renovasi'
 'jelek']


---
## Imputasi Missing Values

In [30]:
# Imputasi median untuk kolom numerik (distribusi bisa skewed)
median_luas   = df['luas_m2'].median()
median_harga  = df['harga_juta'].median()
modus_kamar   = df['kamar'].mode()[0]

print(f'Nilai imputasi luas_m2  : {median_luas} (median)')
print(f'Nilai imputasi harga_juta: {median_harga} (median)')
print(f'Nilai imputasi kamar    : {modus_kamar} (modus)')

# Lakukan imputasi
df['luas_m2']    = df['luas_m2'].fillna(median_luas)
df['harga_juta'] = df['harga_juta'].fillna(median_harga)
df['kamar']      = df['kamar'].fillna(modus_kamar)

print(f'\nMissing values setelah imputasi: {df.isnull().sum().sum()}')

Nilai imputasi luas_m2  : 193.8 (median)
Nilai imputasi harga_juta: 655.0 (median)
Nilai imputasi kamar    : 1.0 (modus)

Missing values setelah imputasi: 0


---
## Tangani Outlier dengan IQR Fence

In [31]:
# Fungsi deteksi outlier IQR
def deteksi_outlier_iqr(df, kolom):
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[kolom] < lower) | (df[kolom] > upper)]
    return lower, upper, outliers

# Deteksi & tangani outlier untuk kolom target
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    lower, upper, outliers = deteksi_outlier_iqr(df, col)
    print(f'[{col}] — Batas IQR: [{lower:.2f}, {upper:.2f}] | Outlier: {len(outliers)} baris')

    # Clip (capping) nilai outlier ke batas IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

print('\nOutlier berhasil ditangani dengan IQR Fence (capping).')

[harga_juta] — Batas IQR: [-422.75, 1719.25] | Outlier: 3 baris
[luas_m2] — Batas IQR: [-145.22, 512.97] | Outlier: 1 baris
[tahun_bangun] — Batas IQR: [1960.50, 2042.50] | Outlier: 3 baris

Outlier berhasil ditangani dengan IQR Fence (capping).


In [32]:
# Verifikasi statistik setelah capping
print('=== STATISTIK SETELAH PENANGANAN OUTLIER ===')
df[['harga_juta', 'luas_m2', 'tahun_bangun']].describe()

=== STATISTIK SETELAH PENANGANAN OUTLIER ===


,harga_juta,luas_m2,tahun_bangun
count,130.000000,130.000000,130.000000
mean,686.490385,188.274423,2001.542308
std,404.633957,95.297150,13.505818
min,-422.750000,-50.000000,1960.500000
25%,380.500000,101.600000,1991.250000
50%,655.000000,193.800000,2002.000000
75%,916.000000,266.150000,2011.750000
max,1719.250000,512.975000,2042.500000


---
## Validasi Akhir & Ekspor Dataset Bersih

In [33]:
# Validasi: tidak boleh ada missing values atau duplikat
print('=== VALIDASI AKHIR ===')

total_missing = df.isnull().sum().sum()
total_dup     = df.duplicated().sum()

print(f'Total missing values : {total_missing}  ✓' if total_missing == 0 else f'Total missing values : {total_missing}  ✗ GAGAL!')
print(f'Total duplikat       : {total_dup}  ✓' if total_dup == 0 else f'Total duplikat       : {total_dup}  ✗ GAGAL!')
print(f'Shape akhir          : {df.shape}')

# Assert untuk memastikan kedua kondisi terpenuhi
assert total_missing == 0, 'Masih ada missing values!'
assert total_dup == 0,     'Masih ada duplikat!'

print('\nSemua validasi LULUS!')

=== VALIDASI AKHIR ===
Total missing values : 0  ✓
Total duplikat       : 0  ✓
Shape akhir          : (130, 7)

Semua validasi LULUS!


In [34]:
# Ekspor dataset bersih
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan sebagai housing_clean.csv')

# Preview hasil akhir
print('\n=== PREVIEW DATASET BERSIH (5 baris pertama) ===')
df.head()

Dataset bersih tersimpan sebagai housing_clean.csv

=== PREVIEW DATASET BERSIH (5 baris pertama) ===


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000.0,baik
1,2,254.0,761.0,Medan,1.0,1995.0,bagus
2,3,249.7,895.0,Depok,1.0,1983.0,baik
3,4,49.7,178.0,Ygy,5.0,2013.0,baik
4,5,133.4,424.0,Medan,5.0,2004.0,sedang


---
## Akses REST API JSONPlaceholder

In [35]:
# Akses API JSONPlaceholder — endpoint /users
URL_USERS = 'https://jsonplaceholder.typicode.com/users'

try:
    response = requests.get(URL_USERS, timeout=10)

    if response.status_code == 200:
        data_users = response.json()
        df_users   = json_normalize(data_users, sep='_')
        print(f'Berhasil mengakses API! Status: {response.status_code}')
        print(f'Jumlah data: {len(df_users)} baris, {len(df_users.columns)} kolom')
    else:
        print(f'Gagal akses API. Status code: {response.status_code}')

except requests.exceptions.ConnectionError as e:
    print(f'Error koneksi: {e}')
except requests.exceptions.Timeout:
    print('Request timeout — server tidak merespons dalam 10 detik')

Berhasil mengakses API! Status: 200
Jumlah data: 10 baris, 15 kolom


In [36]:
# Tampilkan kolom yang tersedia
print('Kolom yang tersedia:')
print(df_users.columns.tolist())

Kolom yang tersedia:
['id', 'name', 'username', 'email', 'phone', 'website', 'address_street', 'address_suite', 'address_city', 'address_zipcode', 'address_geo_lat', 'address_geo_lng', 'company_name', 'company_catchPhrase', 'company_bs']


In [37]:
# Tampilkan kolom-kolom yang relevan
kolom_pilihan = ['id', 'name', 'username', 'email', 'address_city', 'company_name']
print('=== DATA USERS DARI JSONPLACEHOLDER API ===')
df_users[kolom_pilihan]

=== DATA USERS DARI JSONPLACEHOLDER API ===


,id,name,username,email,address_city,company_name
0,1,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,South Christy,Considine-Lockman
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,Howemouth,Johns Group
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,Aliyaview,Abernathy Group
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,Bartholomebury,Yost and Sons
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,Lebsackbury,Hoeger LLC


## Kesimpulan

### Apa yang Dipelajari
Pada pertemuan ketiga ini, saya mempelajari proses data cleaning secara menyeluruh, mulai dari identifikasi masalah kualitas data (missing values, duplikat, inkonsistensi string, dan outlier) hingga penanganannya menggunakan Pandas dan SciPy. Selain itu, saya juga mempelajari cara mengekstrak data dari REST API publik menggunakan library `requests` dan mengonversi respons JSON ke DataFrame Pandas.

### Temuan Utama
- Dataset `housing_dirty.csv` mengandung beberapa masalah kualitas: missing values pada kolom `luas_m2`, `harga_juta`, dan `kamar`; baris duplikat; inkonsistensi penulisan kota (misalnya `jogja` vs `Jogja`); serta outlier pada kolom `harga_juta` dan `tahun_bangun`.
- Imputasi dengan median lebih robust untuk kolom numerik yang memiliki distribusi skewed seperti `harga_juta`, karena tidak terpengaruh nilai ekstrem.
- Metode IQR Fence efektif untuk mendeteksi outlier tanpa terpengaruh oleh outlier itu sendiri, berbeda dengan Z-Score yang sensitif terhadap mean dan standar deviasi.
- API JSONPlaceholder berhasil diakses dan responsnya berhasil dinormalisasi menjadi DataFrame dengan kolom terstruktur seperti `name`, `email`, `address_city`, dan `company_name`.
- Setelah seluruh pipeline cleaning dijalankan, dataset berhasil memenuhi kondisi valid: tidak ada missing values dan tidak ada duplikat.

### Keterbatasan & Pertanyaan yang Muncul
- Bagaimana cara terbaik menyimpan API key dengan aman di Google Colab agar tidak ter-expose saat notebook di-push ke GitHub?